# Day 5: 训练与采样复现

## 🎯 本节目标

完成今天的学习后，你将能够：

1. ✅ 理解完整的训练流程
2. ✅ 运行训练并监控进度
3. ✅ 使用训练好的模型进行采样
4. ✅ 分析生成结果

---

## Part 1: 训练流程深度解析

### 1.1 train.py 完整流程

```python
# train.py 的完整执行流程

def main():
    # 1. 设置随机种子（保证可复现）
    set_seed(config.seed)
    
    # 2. 初始化 W&B（实验跟踪）
    wandb.init(project="Marionette", config=config)
    
    # 3. 实例化数据模块
    datamodule = configs.instantiate_datamodule(config.data)
    datamodule.setup()
    
    # 4. 实例化模型（时间 + 空间）
    tpp_model, discrete_diffusion = configs.instantiate_model(
        config.model, 
        datamodule
    )
    
    # 5. 实例化训练任务
    task = configs.instantiate_task(
        config.task,
        tpp_model, 
        discrete_diffusion
    )
    
    # 6. 设置回调函数
    callbacks = [
        ModelCheckpoint(monitor='val_loss'),
        EarlyStopping(monitor='val_loss', patience=10)
    ]
    
    # 7. 创建 Trainer
    trainer = pl.Trainer(
        max_epochs=config.max_epochs,
        callbacks=callbacks,
        accelerator='gpu',
        devices=1
    )
    
    # 8. 开始训练
    trainer.fit(task, datamodule)
    
    # 9. 保存最终模型
    trainer.save_checkpoint("final_model.ckpt")
```

### 1.2 训练循环详细步骤

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

print("🔥 训练循环详细步骤")
print("="*50)

# 模拟训练流程
class SimpleTrainingLoop:
    """简化的训练循环演示"""
    
    def __init__(self, model, dataloader, optimizer, device):
        self.model = model
        self.dataloader = dataloader
        self.optimizer = optimizer
        self.device = device
    
    def train_epoch(self, epoch):
        """训练一个 epoch"""
        self.model.train()
        total_loss = 0
        
        for batch_idx, batch in enumerate(self.dataloader):
            # 1. 移动数据到设备
            batch = batch.to(self.device)
            
            # 2. 前向传播
            output, loss = self.forward_pass(batch)
            
            # 3. 反向传播
            self.optimizer.zero_grad()
            loss.backward()
            
            # 4. 更新参数
            self.optimizer.step()
            
            # 5. 记录损失
            total_loss += loss.item()
            
            if batch_idx % 100 == 0:
                print(f"  Epoch {epoch}, Batch {batch_idx}, Loss: {loss.item():.4f}")
        
        avg_loss = total_loss / len(self.dataloader)
        return avg_loss
    
    def forward_pass(self, batch):
        """前向传播（在 Marionette 中调用两个模型）"""
        # 时间模型前向传播
        temporal_output = self.model.tpp_model(batch)
        
        # 空间模型前向传播
        spatial_output = self.model.discrete_diffusion(batch)
        
        # 计算总损失
        temporal_loss = self.compute_temporal_loss(temporal_output, batch)
        spatial_loss = self.compute_spatial_loss(spatial_output, batch)
        
        total_loss = temporal_loss + spatial_loss
        return output, total_loss
    
    def compute_temporal_loss(self, output, batch):
        """计算时间损失（负对数似然）"""
        return output['loss']
    
    def compute_spatial_loss(self, output, batch):
        """计算空间损失（交叉熵）"""
        return output['loss']

print("训练流程说明:")
print("\n1. 数据加载: DataModule 提供 DataLoader")
print("2. 批次处理: 每个批次包含多条轨迹")
print("3. 双模型训练: 时间模型 + 空间模型")
print("4. 损失计算: 时间损失 + 空间损失")
print("5. 梯度更新: 两个模型分别更新参数")

### 1.3 W&B 实验跟踪

**Weights & Biases (W&B)** 是一个强大的实验跟踪工具。

In [ ]:
print("📊 W&B 实验跟踪")
print("="*50)

# W&B 使用示例
wandb_usage = """
# 1. 初始化
import wandb

wandb.init(
    project="marionette",
    config={
        "batch_size": 512,
        "learning_rate": 0.001,
        "epochs": 100,
        "model": "Marionette"
    }
)

# 2. 训练过程中记录指标
for epoch in range(epochs):
    train_loss = train_one_epoch()
    val_loss = validate()
    
    # 记录到 W&B
    wandb.log({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "lr": optimizer.param_groups[0]['lr']
    })

# 3. 记录自定义图表
wandb.log({
    "confusion_matrix": wandb.plot.confusion_matrix(...)
})

# 4. 保存模型
wandb.save("model.ckpt")

# 5. 结束
wandb.finish()
"""

print(wandb_usage)

print("\n优势:")
print("- 实时查看训练曲线")
print("- 对比不同实验")
print("- 自动保存模型和日志")
print("- 可视化生成结果")

---

## Part 2: 采样流程详解

### 2.1 sample.py 完整流程

```python
# sample.py 的完整执行流程

def main(run_id):
    # 1. 从 W&B 下载模型
    api = wandb.Api()
    run = api.run(f"marionette/{run_id}")
    model_file = run.file("model.ckpt").download()
    
    # 2. 加载模型
    task = DensityEstimation.load_from_checkpoint(model_file)
    task.eval()
    
    # 3. 加载测试数据
    datamodule = configs.instantiate_datamodule(config.data)
    test_loader = datamodule.test_dataloader()
    
    # 4. 采样循环
    all_samples = []
    for batch in test_loader:
        # 4.1 生成时间样本
        time_samples = task.tpp_model.sample(
            batch_size=len(batch),
            x_n=batch  # 使用真实数据作为条件
        )
        
        # 4.2 生成 POI 序列
        poi_samples = task.discrete_diffusion.sample_fast(
            time_samples
        )
        
        # 4.3 合并结果
        samples = combine_samples(time_samples, poi_samples)
        all_samples.append(samples)
    
    # 5. 保存结果
    torch.save(all_samples, "generated_sequences.pkl")
```

### 2.2 采样详细步骤

In [ ]:
print("🎲 采样流程详细步骤")
print("="*50)

sampling_steps = [
    ("1", "加载模型", "从检查点恢复训练好的模型"),
    ("2", "准备条件", "从测试集获取条件（或自定义条件）"),
    ("3", "初始化", "从纯噪声或 HPP 开始"),
    ("4", "时间采样", "运行 Add-THIN 反向扩散"),
    ("5", "空间采样", "运行离散扩散反向采样"),
    ("6", "后处理", "去除 padding，合并结果"),
    ("7", "保存", "保存为 .pkl 或 JSON 格式"),
]

for step, name, desc in sampling_steps:
    print(f"{step}. {name}")
    print(f"   {desc}\n")

print("关键点:")
print("- 时间采样和空间采样是分开的")
print("- 可以使用真实数据作为条件（条件生成）")
print("- 也可以完全从噪声生成（无条件生成）")

---

## Part 3: 实际运行训练

### 3.1 准备工作

#### 步骤 1: 检查数据

In [ ]:
import os
import torch

print("📁 检查数据和配置")
print("="*50)

# 检查数据文件
data_dir = "data/Istanbul"
train_file = f"{data_dir}/Istanbul_train.pkl"
test_file = f"{data_dir}/Istanbul_test.pkl"

print(f"数据目录: {data_dir}")
print(f"训练文件: {'✓' if os.path.exists(train_file) else '✗'}")
print(f"测试文件: {'✓' if os.path.exists(test_file) else '✗'}")

if os.path.exists(train_file):
    # 加载并检查数据
    data = torch.load(train_file, map_location='cpu')
    print(f"\n数据集信息:")
    print(f"  序列数量: {len(data['sequences'])}")
    print(f"  POI 类别数: {data['num_marks']}")
    print(f"  POI 总数: {data['num_pois']}")
    print(f"  最大时间: {data['t_max']} 秒")

# 检查配置文件
config_files = [
    "config/train.yaml",
    "config/model/Marionette.yaml",
    "config/task/density.yaml",
]

print(f"\n配置文件:")
for f in config_files:
    exists = '✓' if os.path.exists(f) else '✗'
    print(f"  {exists} {f}")

#### 步骤 2: 启动训练

```bash
# 基础训练命令
python train.py

# 指定配置
python train.py --config-name train

# 覆盖参数
python train.py data.batch_size=256 trainer.max_epochs=50
```

### 3.2 监控训练进度

训练时关注以下指标：

In [ ]:
print("📊 训练指标监控")
print("="*50)

metrics = [
    ("train_loss", "训练损失", "应该持续下降"),
    ("val_loss", "验证损失", "应该与 train_loss 一起下降"),
    ("temporal_loss", "时间损失", "Add-THIN 模型的损失"),
    ("spatial_loss", "空间损失", "离散扩散模型的损失"),
    ("epoch", "训练轮数", "当前训练到第几轮"),
    ("lr", "学习率", "优化器的学习率"),
]

for key, name, desc in metrics:
    print(f"\n【{name}】")
    print(f"  W&B 名称: {key}")
    print(f"  说明: {desc}")

print("\n" + "="*50)
print("正常训练的标志:")
print("✓ train_loss 稳定下降")
print("✓ val_loss 不会持续上升（过拟合）")
print("✓ temporal_loss 和 spatial_loss 都在下降")
print("✓ 损失不会变成 NaN")

print("\n异常情况:")
print("✗ loss 变成 NaN → 学习率过大或数据异常")
print("✗ val_loss 上升 → 过拟合，需要早停")
print("✗ loss 不下降 → 模型或数据有问题")

---

## Part 4: 运行采样

### 4.1 从 W&B 加载模型

```bash
# 查看运行记录
wandb login
wandb online  # 查看所有运行

# 使用 run_id 采样
sh sample_evaluation.sh <run_id>
```

In [ ]:
print("🎯 采样命令")
print("="*50)

print("\n方法 1: 使用 sample.py")
print("```bash")
python sample.py --run_id <wandb_run_id>")
print("```")

print("\n方法 2: 使用 sample_evaluation.sh")
print("```bash")
sh sample_evaluation.sh <wandb_run_id>")
print("```")

print("\n采样参数说明:")
print("--run_id: W&B 运行 ID（必需）")
print("--num_samples: 生成轨迹数量（默认: 测试集大小）")
print("--output: 输出路径（默认: outputs/）")

print("\n输出文件:")
print("- generated_sequences.pkl: 生成的轨迹数据")
print("- evaluation_results.json: 评估结果")

### 4.2 分析生成结果

In [ ]:
# 模拟分析生成结果
print("📊 分析生成结果")
print("="*50)

print("\n统计指标:")
metrics_analysis = [
    "平均轨迹长度",
    "时间间隔分布",
    "POI 类别分布",
    "访问频率最高的 POI",
    "时间覆盖范围",
]

for metric in metrics_analysis:
    print(f"  - {metric}")

print("\n对比分析:")
print("  - 真实数据 vs 生成数据的统计分布")
print("  - 评估下游任务性能（LocRec, NexLoc 等）")

print("\n可视化:")
print("  - 时间分布直方图")
print("  - POI 热力图")
print("  - 轨迹可视化（使用 Deck.gl）")

---

## Part 5: 常见问题与解决

### Q1: 训练时显存不足

**解决方案**:
```yaml
# config/train.yaml
data:
  batch_size: 128  # 减小 batch_size
trainer:
  accumulate_grad_batches: 4  # 梯度累积
```

In [ ]:
print("❓ 常见问题速查")
print("="*50)

solutions = {
    "CUDA OOM": [
        "1. 减小 batch_size",
        "2. 使用梯度累积",
        "3. 减少扩散步数"
    ],
    "Loss NaN": [
        "1. 降低学习率",
        "2. 检查数据是否有异常值",
        "3. 使用梯度裁剪"
    ],
    "Loss 不下降": [
        "1. 检查数据加载是否正确",
        "2. 检查模型结构",
        "3. 调整学习率"
    ],
    "生成质量差": [
        "1. 训练更多 epochs",
        "2. 调整模型超参数",
        "3. 增加数据量"
    ]
}

for problem, sols in solutions.items():
    print(f"\n【{problem}】")
    for sol in sols:
        print(f"  {sol}")

---

## 📝 Day 5 总结

### 今天学到了什么？

1. **训练流程**: 完整的训练循环和 W&B 跟踪
2. **采样流程**: 从模型加载到生成轨迹
3. **监控技巧**: 关注哪些指标，如何判断训练状态
4. **问题解决**: 常见问题的快速解决方案

---

### 🎯 明天预告：后端 API 实现

Day 6 将学习：

1. FastAPI 基础
2. 实现完整的推理 API
3. 数据格式转换
4. API 测试和文档

---

**🎉 恭喜完成 Day 5！**

明天我们将构建完整的后端 API，为前端可视化做准备。